# Phase 3 — Agentic Layer (Analyst & Executive Challenger)

> **Context**: This notebook wraps the existing Decision Engine V2.1 to produce **strategy‑ready** outputs with two agents: an **Analyst Agent** (facts & KPIs) and an **Executive Challenger Agent** (stress‑tests & robustness).

**Pre‑requisites**
- You already have the Decision Engine code available (e.g., `compare_scenarios`, `run_scenario`, `greedy_allocate`) from your `02_decision_engine_v2_1_simple` work.
- VS Code + Python 3.14 in a venv.

**What you get here**
- Minimal, credible agents (no heavy framework).
- A compact JSON payload you can directly pass to a Decision Room UI or a slide deck.

*Language: comments are in English; headings include French guidance where useful.*


In [ ]:
# − Environment check (optional)
import sys, platform
print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
try:
    import pandas as pd, numpy as np
    import matplotlib, seaborn
    print('pandas:', pd.__version__)
    print('numpy:', np.__version__)
    print('matplotlib:', matplotlib.__version__)
    print('seaborn:', seaborn.__version__)
except Exception as e:
    print('[Warn] Some libraries not importable:', e)


## Imports & adapters — branch to your Decision Engine

**Goal**: re‑use your existing functions without duplicating logic.

We try 3 options (in order):
1. Import from a local module (e.g., `decision_engine_v2_1_simple.py` or `decision_engine.py`).
2. Import from a package path you configured.
3. As a last resort, instruct you to export the key functions from your notebook to a `.py` module.


In [ ]:
# − Try to import your engine. Adjust the module name to your setup if needed.
ENGINE_IMPORT_OK = False
run_scenario = compare_scenarios = greedy_allocate = None

candidate_modules = [
    'decision_engine_v2_1_simple',
    'decision_engine',
    'src.decision_engine_v2_1_simple',
    'src.decision_engine'
]

for mod in candidate_modules:
    try:
        m = __import__(mod, fromlist=['*'])
        run_scenario = getattr(m, 'run_scenario', None)
        compare_scenarios = getattr(m, 'compare_scenarios', None)
        greedy_allocate = getattr(m, 'greedy_allocate', None)
        if all([run_scenario, compare_scenarios, greedy_allocate]):
            print(f'[OK] Imported Decision Engine from: {mod}')
            ENGINE_IMPORT_OK = True
            break
    except Exception:
        pass

if not ENGINE_IMPORT_OK:
    print('[Action needed] Could not import your Decision Engine.\n'
          'Please export the functions `run_scenario`, `compare_scenarios`, `greedy_allocate` '
          'into a module (e.g., decision_engine.py) located next to this notebook, then re-run this cell.')


## Agents — Analyst & Executive Challenger

**Design principles**
- Keep it simple and defensible (no complex multi-agent loops).
- Reuse your guardrails: uplift caps, decay, payback gate, AU budget.
- Output compact JSON for UI/Deck.


In [ ]:
# Analyst Agent: synthesizes facts & KPIs into an executive package
from typing import Dict, Any, List

class AnalystAgent:
    def __init__(self, compare_func, greedy_func):
        self.compare = compare_func
        self.greedy = greedy_func

    def summarize(self, policies: Dict[str, Dict[str, Any]], greedy_budget_au: float = 50.0) -> Dict[str, Any]:
        if self.compare is None or self.greedy is None:
            raise RuntimeError('Decision Engine functions not available. Import them first.')
        cmp_df = self.compare(policies)
        greedy_summary = self.greedy(budget_au=greedy_budget_au)

        kpi_deltas: List[Dict[str, Any]] = []
        for name, row in cmp_df.iterrows():
            kpi_deltas.append({
                'option': name,
                'delta_net_eur': float(row.get('delta_net_eur', 0.0)),
                'delta_grr_pp':  float(row.get('delta_grr_pp', 0.0)),
                'delta_nrr_pp':  float(row.get('delta_nrr_pp', 0.0)),
                'effort_au':     float(row.get('effort_au', 0.0))
            })

        executive_takeaway = self._make_takeaway(kpi_deltas, greedy_summary)

        out = {
            'executive_takeaway': executive_takeaway,
            'kpi_deltas': kpi_deltas,
            'greedy_summary': {
                'budget_au': float(greedy_summary.get('budget_used_au', 0.0)),
                'delta_net_eur': float(greedy_summary.get('delta_net_eur', 0.0)),
                'delta_grr_pp': float(greedy_summary.get('delta_grr_pp', 0.0))
            },
            'watchouts': [
                'ROI = ROI marginal local (non extrapolable)',
                'Effort (AU) != budget financier; AU->EUR is a simplification'
            ]
        }
        return out

    def _make_takeaway(self, kpi_deltas: List[Dict[str, Any]], greedy_summary: Dict[str, Any]) -> str:
        if not kpi_deltas:
            return 'No scenario deltas available.'
        best = None
        best_score = -1.0
        for x in kpi_deltas:
            denom = x['effort_au'] if x['effort_au'] else 1e-6
            score = x['delta_net_eur'] / denom
            if score > best_score:
                best, best_score = x, score
        gnet = int(greedy_summary.get('delta_net_eur', 0))
        return (f"Under constraints, {best["option"]} maximizes EUR per AU; "
                f"greedy plan unlocks ~EUR{gnet:,} within budget.")


In [ ]:
# Executive Challenger Agent: runs stress-tests & emits robustness verdicts
from typing import List, Dict, Any
import pandas as pd

class ChallengerAgent:
    def __init__(self, run_func, compare_func, greedy_func):
        self.run = run_func
        self.compare = compare_func
        self.greedy = greedy_func

    def stress_test(self, base_policies: Dict[str, Dict[str, Any]], tests: List[Dict[str, Any]]):
        if self.compare is None or self.greedy is None:
            raise RuntimeError('Decision Engine functions not available. Import them first.')
        results = []
        for t in tests:
            pol_mod = self._apply_policy_mod(base_policies, t.get('policy_mod', {}))
            cmp_df = self.compare(pol_mod)
            verdict = self._verdict(cmp_df)
            out = {
                'name': t.get('name', 'unnamed test'),
                'verdict': verdict,
                'result_table': cmp_df.to_dict(orient='index')
            }
            if 'greedy_budget_au' in t and t['greedy_budget_au'] is not None:
                out['greedy_summary'] = self.greedy(budget_au=float(t['greedy_budget_au']))
            results.append(out)
        robust_pick = self._pick_robust(results)
        return {'stress_tests': results, 'robust_recommendation': robust_pick}

    def _apply_policy_mod(self, base: Dict[str, Dict[str, Any]], mod: Dict[str, Any]):
        # Shallow merge: override nested dicts when provided
        out = {k: (v.copy() if isinstance(v, dict) else v) for k, v in base.items()}
        for pol_name, changes in mod.items():
            cur = out.get(pol_name, {})
            if isinstance(cur, dict) and isinstance(changes, dict):
                tmp = cur.copy()
                tmp.update(changes)
                out[pol_name] = tmp
            else:
                out[pol_name] = changes
        return out

    def _verdict(self, cmp_df: pd.DataFrame) -> Dict[str, str]:
        verdict = {}
        for name, row in cmp_df.iterrows():
            holds = (row.get('delta_net_eur', 0) > 0) and \n                    (row.get('delta_grr_pp', 0) >= -0.05) and \n                    (row.get('delta_nrr_pp', 0) >= -0.05)
            verdict[name] = 'holds' if holds else 'breaks'
        return verdict

    def _pick_robust(self, tests: List[Dict[str, Any]]):
        counts: Dict[str, int] = {}
        for t in tests:
            for opt, v in t['verdict'].items():
                counts.setdefault(opt, 0)
                counts[opt] += (1 if v == 'holds' else 0)
        if not counts:
            return {'option': None, 'rationale': 'no tests'}
        best = max(counts.items(), key=lambda x: x[1])[0]
        return {'option': best, 'rationale': "most frequent 'holds' across stress tests"}


## Policies (examples) — align with your existing scenarios

Replace these with your exact policy dicts if you already defined them in your engine notebook/module.


In [ ]:
# Example policies (to be aligned with your existing ones)
policies = {
    'Retention-first': {
        'segment': {'Enterprise': {'churn_bps': 150}},
        'cell': {('EMEA', 'Mid-Market'): {'churn_bps': 150}}
    },
    'Balanced Growth': {
        'region': {'EMEA': {'new_pct': 0.06}},
        'segment': {'SMB': {'new_pct': 0.04, 'react_pct': 0.10}}
    },
    'Protect Whales': {
        'cell': {('APAC', 'Enterprise'): {'churn_bps': 100, 'react_pct': 0.05}}
    }
}
policies


## Stress tests — standard set (adjust as needed)


In [ ]:
# Minimal, high-signal stress tests
stress_tests = [
    {'name': 'Expansion -30%', 'policy_mod': {'Balanced Growth': {'react_pct': 0.7}}},
    {'name': 'Churn +100 bps on Enterprise', 'policy_mod': {'Retention-first': {'churn_bps': 100}}},
    {'name': 'Budget cut 50%', 'greedy_budget_au': 25},
]
stress_tests


## Run — produce the agentic payload (JSON)

If your engine was imported correctly, the cells below will create two outputs:
- `analyst_pack`: executive summary of KPIs and greedy plan
- `challenge_pack`: stress tests results and robust recommendation


In [ ]:
analyst_pack = challenge_pack = None
if ENGINE_IMPORT_OK:
    analyst = AnalystAgent(compare_func=compare_scenarios, greedy_func=greedy_allocate)
    analyst_pack = analyst.summarize(policies, greedy_budget_au=50.0)

    challenger = ChallengerAgent(run_func=run_scenario, compare_func=compare_scenarios, greedy_func=greedy_allocate)
    challenge_pack = challenger.stress_test(policies, stress_tests)
else:
    print('[Skip] Engine not imported — complete the import step above and re-run.')

analyst_pack, challenge_pack


## Save payload — for Decision Room UI or deck consumption


In [ ]:
import json, datetime, pathlib
payload_path = pathlib.Path('agentic_output.json')
if analyst_pack is not None and challenge_pack is not None:
    payload = {'analyst': analyst_pack, 'challenger': challenge_pack, 'generated_at': datetime.datetime.utcnow().isoformat() + 'Z'}
    payload_path.write_text(json.dumps(payload, indent=2))
    print(f'[OK] Saved: {payload_path.resolve()}')
else:
    print('[Skip] Nothing saved — run the previous cell after fixing the import.')


---
### Next steps (quick)
- (Optionnel) Ajoute 2 stress tests supplémentaires: Payback gate 12 mois; DECAY = 0.75.
- Alimente ta *Decision Room* avec `agentic_output.json`.
- Pour le deck: mappe `executive_takeaway`, le tableau `kpi_deltas` et `robust_recommendation`.
